
# 3D Brownian Motion Sparse Dots Dataset (260×346×200, 100 dots @ 1000 fps)

This notebook creates a sparse 3D dataset of moving dots undergoing Brownian motion.
It avoids massive memory usage by storing data in an **event list** rather than dense frames.


## Parameters

In [ ]:

# ---- Simulation Parameters ----
# Volume size (Z, Y, X) — feel free to reorder if you prefer (X, Y, Z).
Z, Y, X = 200, 260, 346

num_dots = 100            # number of 1×1×1 dots
fps = 1000                # frames per second
num_frames = 1000         # total frames (e.g., 1 second at 1000 fps)

# Brownian motion settings
# Either specify per-frame Gaussian step std (voxels) directly...
per_frame_step_std = (0.9, 0.9, 0.9)  # (σz, σy, σx) in voxels per frame
# ...or derive from a diffusion coefficient D via σ = sqrt(2*D*Δt). For simplicity, we'll use the σ above.

# RNG seed for reproducibility
seed = 42

# Output file (sparse events)
out_path = "brownian_3d_dots_events.npz"

# Sanity-check visualization options
num_trajectories_to_plot = 5  # number of dot trajectories to visualize
z_slices_to_preview = [0, Z//4, Z//2, 3*Z//4, Z-1]  # which z-slices to preview for a given frame
frame_for_slice_preview = 0    # which frame to preview slice occupancy

print(f"Configured volume (Z,Y,X)=({Z},{Y},{X}), dots={num_dots}, frames={num_frames}, fps={fps}")


## Utilities

In [ ]:

import numpy as np

def reflect_bounds(pos, low, high):
    """Reflecting boundary for pos in [low, high).
    Supports vectorized arrays.
    """
    span = high - low
    # Bring into an infinite tiling and reflect every other tile
    # Convert to distance from low
    x = pos - low
    # mod in 2*span, then reflect
    m = np.mod(x, 2*span)
    reflected = np.where(m < span, m, 2*span - m)
    return reflected + low

def initialize_positions(num_dots, Z, Y, X, rng):
    """Initialize *float* positions uniformly in the volume (centered within voxels)."""
    z = rng.uniform(0, Z, size=num_dots)
    y = rng.uniform(0, Y, size=num_dots)
    x = rng.uniform(0, X, size=num_dots)
    return np.stack([z, y, x], axis=1).astype(np.float64)

def simulate_brownian_sparse(Z, Y, X, num_dots, num_frames, per_frame_step_std, seed=0):
    """Simulate Brownian motion with reflecting boundaries.
    
    Returns:
        t_idx: (N,) int32 frame indices
        zyx:   (N,3) float32 positions (continuous)
        ijk:   (N,3) int32 voxel indices (z,y,x) after rounding
        ids:   (N,) int32 dot IDs [0..num_dots-1]
    Where N = num_dots * num_frames
    """
    rng = np.random.default_rng(seed)
    pos = initialize_positions(num_dots, Z, Y, X, rng)
    stdz, stdy, stdx = per_frame_step_std

    # Preallocate outputs
    N = num_dots * num_frames
    t_idx = np.empty(N, dtype=np.int32)
    zyx = np.empty((N, 3), dtype=np.float32)
    ijk = np.empty((N, 3), dtype=np.int32)
    ids = np.empty(N, dtype=np.int32)

    # Simulate
    write_ptr = 0
    for t in range(num_frames):
        # Record current positions
        t_idx[write_ptr:write_ptr+num_dots] = t
        zyx[write_ptr:write_ptr+num_dots] = pos.astype(np.float32)
        # Round to voxel indices safely
        z_ = np.clip(np.rint(pos[:, 0]).astype(np.int32), 0, Z-1)
        y_ = np.clip(np.rint(pos[:, 1]).astype(np.int32), 0, Y-1)
        x_ = np.clip(np.rint(pos[:, 2]).astype(np.int32), 0, X-1)
        ijk[write_ptr:write_ptr+num_dots, 0] = z_
        ijk[write_ptr:write_ptr+num_dots, 1] = y_
        ijk[write_ptr:write_ptr+num_dots, 2] = x_
        ids[write_ptr:write_ptr+num_dots] = np.arange(num_dots, dtype=np.int32)
        write_ptr += num_dots

        # Step: independent Gaussian increments per axis
        dZ = rng.normal(0.0, stdz, size=num_dots)
        dY = rng.normal(0.0, stdy, size=num_dots)
        dX = rng.normal(0.0, stdx, size=num_dots)
        pos[:, 0] += dZ
        pos[:, 1] += dY
        pos[:, 2] += dX

        # Reflect at boundaries
        pos[:, 0] = reflect_bounds(pos[:, 0], 0.0, float(Z))
        pos[:, 1] = reflect_bounds(pos[:, 1], 0.0, float(Y))
        pos[:, 2] = reflect_bounds(pos[:, 2], 0.0, float(X))

    return t_idx, zyx, ijk, ids


## Run Simulation

In [ ]:

t_idx, zyx, ijk, ids = simulate_brownian_sparse(
    Z=Z, Y=Y, X=X,
    num_dots=num_dots,
    num_frames=num_frames,
    per_frame_step_std=per_frame_step_std,
    seed=seed
)
print("Simulation complete.")
print(f"Events: {len(t_idx):,} rows (should be num_dots × num_frames = {num_dots*num_frames:,}).")


## Quick Visual Sanity Checks

In [ ]:

# Matplotlib is used with a single plot per chart and no explicit color settings.
import matplotlib.pyplot as plt
import numpy as np

# Plot a few trajectories (z vs time, y vs time, x vs time — separate plots for clarity)
sel_ids = np.arange(min(num_trajectories_to_plot, num_dots))
figures = []

# z(t) for a few dots
plt.figure()
for did in sel_ids:
    mask = ids == did
    plt.plot(t_idx[mask], zyx[mask, 0])
plt.title("Trajectories: z vs frame")
plt.xlabel("frame")
plt.ylabel("z")
figures.append(plt.gcf())

# y(t) for a few dots
plt.figure()
for did in sel_ids:
    mask = ids == did
    plt.plot(t_idx[mask], zyx[mask, 1])
plt.title("Trajectories: y vs frame")
plt.xlabel("frame")
plt.ylabel("y")
figures.append(plt.gcf())

# x(t) for a few dots
plt.figure()
for did in sel_ids:
    mask = ids == did
    plt.plot(t_idx[mask], zyx[mask, 2])
plt.title("Trajectories: x vs frame")
plt.xlabel("frame")
plt.ylabel("x")
figures.append(plt.gcf())

# Occupancy preview on selected z-slices for one frame
frame = int(frame_for_slice_preview)
mask_f = t_idx == frame
z_frame = ijk[mask_f, 0]
y_frame = ijk[mask_f, 1]
x_frame = ijk[mask_f, 2]

for zs in z_slices_to_preview:
    plt.figure()
    # Build a sparse 2D occupancy for the slice
    occ = np.zeros((Y, X), dtype=np.uint8)
    on_slice = z_frame == zs
    if np.any(on_slice):
        ys = y_frame[on_slice]
        xs = x_frame[on_slice]
        occ[ys, xs] = 1
    plt.imshow(occ)
    plt.title(f"Occupancy at frame={frame}, z-slice={zs}")
    plt.xlabel("x")
    plt.ylabel("y")
    figures.append(plt.gcf())

print("Rendered sanity-check plots.")


## Save and Load Helpers

In [ ]:

def save_events_npz(path, t_idx, zyx, ijk, ids, meta=None):
    if meta is None:
        meta = {}
    np.savez_compressed(
        path,
        t_idx=t_idx.astype(np.int32),
        zyx=zyx.astype(np.float32),
        ijk=ijk.astype(np.int32),
        ids=ids.astype(np.int32),
        Z=np.array([Z], dtype=np.int32),
        Y=np.array([Y], dtype=np.int32),
        X=np.array([X], dtype=np.int32),
        fps=np.array([fps], dtype=np.int32),
        num_frames=np.array([num_frames], dtype=np.int32),
        **{f"meta_{k}": np.array([v]) for k, v in meta.items()}
    )
    print(f"Saved events to {path}")

def load_events_npz(path):
    d = np.load(path, allow_pickle=True)
    t_idx = d["t_idx"]
    zyx = d["zyx"]
    ijk = d["ijk"]
    ids = d["ids"]
    Z = int(d["Z"][0])
    Y = int(d["Y"][0])
    X = int(d["X"][0])
    fps = int(d["fps"][0])
    num_frames = int(d["num_frames"][0])
    meta = {k[5:]: d[k][0] for k in d.files if k.startswith("meta_")}
    return (t_idx, zyx, ijk, ids), (Z, Y, X, fps, num_frames), meta

# Save
save_events_npz(out_path, t_idx, zyx, ijk, ids, meta={"note":"3D Brownian sparse dots"})


## Optional: Dense Frame Generator (on the fly)

In [ ]:

def dense_frame_from_events(t, t_idx, ijk, shape):
    """Construct a dense 3D binary frame for time index t from the event list.
    shape = (Z, Y, X)
    NOTE: This allocates Z*Y*X, so use sparingly with large volumes.
    """
    Z, Y, X = shape
    vol = np.zeros((Z, Y, X), dtype=np.uint8)
    mask = t_idx == t
    if np.any(mask):
        coords = ijk[mask]  # (N_t, 3)
        vol[coords[:,0], coords[:,1], coords[:,2]] = 1
    return vol

# Example usage (commented out to avoid large memory for repeated calls):
# vol0 = dense_frame_from_events(0, t_idx, ijk, (Z, Y, X))
# print(vol0.shape, vol0.sum())


## Done


**Files produced**  
- `brownian_3d_dots_events.npz`: Event-format dataset with fields `t_idx, zyx, ijk, ids` and metadata.

You can adjust parameters and re-run to regenerate the dataset.
